# 1. Парадокс дней рождения

В теории вероятностей и комбинаторике с днями рождения связана известная задача, называемая «парадокс дней рождения». Данный «парадокс» гласит, что в любой случайно собранной группе из 23 человек с вероятностью более 50 % у двух или более человек дни рождения совпадут (на первый взгляд, это противоречит житейской интуиции). С ростом численности группы вероятность наличия совпадения дней рождения быстро приближается к единице: так, в случайно собранной группе из 57 человек двое из них имеют дни рождения в один день с вероятностью около 99 %.

Такое утверждение может показаться противоречащим здравому смыслу, так как вероятность одного родиться в определённый день года довольно мала, а вероятность того, что двое родились в конкретный день — ещё меньше, но, чем больше людей в группе, тем больше появляется сравнений между парами людей. Таким образом, оно не является парадоксом в строгом научном смысле — логического противоречия в нём нет, а лишь заключается лишь в различиях между интуитивным восприятием ситуации человеком и результатами математического расчёта.

Источники:
- https://habr.com/ru/articles/72301/
- https://pikabu.ru/story/paradoks_dney_rozhdeniya_175264
- https://ru.wikipedia.org/wiki/Парадокс_дней_рождения


## Задача

Определить вероятность того, что в группе, состоящей из 23 человек, у двух людей будет совпадение дней рождения (число и месяц).

Рассчитаем сначала $\bar p(n)$ — вероятность того, что в группе из $n$ человек дни рождения всех будут различными.

Возьмём наугад **первого** человека из группы и запомним его день рождения. Затем возьмём наугад **второго** человека, при этом вероятность того, что у него день рождения не совпадёт с днём рождения первого, равна

$$1-\frac{1}{365}.$$

Затем возьмём **третьего** человека; при этом вероятность того, что его день рождения не совпадёт с днём рождения одного из первых двух, равна

$$1-\frac{2}{365}.$$

Рассуждая по аналогии, для последнего человека, для которого вероятность несовпадения его дня рождения со всеми предыдущими будет равна

$$1-\frac{n-1}{365}.$$

Перемножая все эти вероятности, получаем вероятность того, что все дни рождения в группе будут различными:

$$
\bar p(n)
=
1\cdot
\left(1-\frac{1}{365}\right)
\cdot
\left(1-\frac{2}{365}\right)
\cdots
\left(1-\frac{n-1}{365}\right)
=
$$

$$
=
\frac{365\cdot364\cdot363\cdots(365-n+1)}{365^n}
=
\frac{365!}{365^n(365-n)!}.
$$

Тогда вероятность того, что хотя бы у двух человек из $n$ дней рождения совпадут, равна

$$
p(n)=1-\bar p(n).
$$

Значение этой функции превосходит $\frac12$ при $n=23$, при этом вероятность совпадения равна примерно $50.73\%$, а $p(22)\approx47.57\%$.


### Значения вероятности

| Значение $n$ | Вероятность $p(n)$ |
|---:|---:|
| 10 | 12% |
| 20 | 41% |
| 30 | 70% |
| 50 | 97% |
| 100 | 99.99996% |
| 200 | 99.99999999999999999999999998% |
| 366 | 100% |

Ниже строится график зависимости вероятности совпадения дней рождения от количества людей в группе.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math import lgamma, log

def probability_birthday_match(n):
    """Вероятность того, что среди n человек есть совпадающие дни рождения."""
    if n >= 365:
        return 1.0

    # Используем логарифмы для избежания переполнения:
    # log(P) = log(365!) - n*log(365) - log((365-n)!)
    log_prob = lgamma(366) - n * np.log(365) - lgamma(366 - n)
    return 1 - np.exp(log_prob)

# Создадим данные для графика
n_values = np.arange(1, 101)
probabilities = [probability_birthday_match(n) for n in n_values]

# Строим график
plt.figure(figsize=(10, 6))
plt.plot(n_values, probabilities, 'b-', linewidth=2)
plt.xlabel('Количество людей (n)')
plt.ylabel('Вероятность совпадений')
plt.title('Задача о днях рождения')
plt.grid(True, alpha=0.3)
plt.axhline(y=0.5, color='r', linestyle='--', alpha=0.7,
            label='Вероятность 50%')
plt.legend()
plt.show()

# Находим, когда вероятность падает/становится больше 50%
for n, prob in zip(n_values, probabilities):
    if prob > 0.5:
        print(f"При n = {n} вероятность совпадений = {prob:.3f} (> 50%)")
        break


## Замечание

При вычислении выражения

$$
\frac{365!}{365^n(365-n)!}
$$

напрямую возникает переполнение из-за очень больших факторов.

Для решения этой проблемы можно использовать следующие подходы:

- использовать логарифмы;
- постепенно вычислять произведение.


In [ ]:
def birthday_probability_product(n):
    """Вычисляет произведение вероятностей."""
    if n >= 365:
        return 1.0

    prob = 1.0
    for i in range(n):
        prob *= (365 - i) / 365

    return 1 - prob


## Смоделируем решение этой задачи в Pandas


In [ ]:
import pandas as pd
import numpy as np


In [ ]:
bd = pd.Series(range(1, 366))


In [ ]:
group = bd.sample(23, replace=True)
group.duplicated()


Отобразим номер дня (номера дней) в году, когда совпали дни рождения:


In [ ]:
print(*group[group.duplicated() == True].values)


Можно просто отобразить информацию о том, были ли совпадения:


In [ ]:
bd.sample(23, replace=True).duplicated().max()


Проведём серию из 10000 испытаний.


In [ ]:
rooms = [bd.sample(23, replace=True).duplicated().max() for _ in range(10_000)]


Определим частоту события, когда были совпадения дней рождений в группе из 23 человек.


In [ ]:
np.mean(rooms)


Определим частоту события, когда были совпадения дней рождений в группе из 50 человек.


In [ ]:
rooms = [bd.sample(50, replace=True).duplicated().max() for _ in range(10_000)]
np.mean(rooms)
